# Polarized ORSO experiment

This notebook demonstrates the **polarization-aware ORSO reader** in EasyReflectometry.

We will:

1. Load a polarized `.ort` file that contains two spin channels (`spin_up` / `spin_down`) using the new `load_polarized` entry point.
2. Construct a model from the sample **description** stored in the ORSO header (via the sample loader).
3. *Try* to fit both spin channels using the Refl1d **magnetic** interface, following the pattern from the `magnetism.ipynb` tutorial.

> **Note on the example file.** `test_example2.ort` is a tiny ORSO *formatting* fixture: it carries only a couple of placeholder data points per channel (with non-physical reflectivities `R > 1`). The goal here is to exercise the **workflow end to end**, not to obtain a physically meaningful fit.

## Setup

In [ ]:
%matplotlib inline

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import refl1d
import refl1d.names
from orsopy.fileio import orso
from scipy.optimize import least_squares

import easyreflectometry
from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.calculators.refl1d.wrapper import _get_polarized_probe
from easyreflectometry.data import load_polarized
from easyreflectometry.model import Model
from easyreflectometry.model import PercentageFwhm
from easyreflectometry.orso_utils import LoadOrso
from easyreflectometry.sample import Layer
from easyreflectometry.sample import Material
from easyreflectometry.sample import Multilayer
from easyreflectometry.sample import Sample

In [ ]:
print(f'numpy: {np.__version__}')
print(f'easyreflectometry: {easyreflectometry.__version__}')
print(f'refl1d: {refl1d.__version__}')

In [ ]:
# Resolve the example file relative to the installed package so this notebook
# runs regardless of the working directory.
DATA_FILE = os.path.join(os.path.dirname(easyreflectometry.__file__), '..', '..', 'tests', '_static', 'test_example2.ort')
DATA_FILE = os.path.abspath(DATA_FILE)
print(DATA_FILE)

## 1. Load the two spin channels

`test_example2.ort` declares `polarization: p` and two datasets labelled `spin_up` and `spin_down`.
Because every dataset carries a recognised `data_set` spin label, the reader can resolve the spin
direction **unequivocally** and returns a `PolarizedData` container instead of a flat data group.

A `PolarizedData` exposes:

* `polarization` &mdash; the classification (`half_polarized` here),
* `spin_by_key` &mdash; the `up`/`down` direction for each dataset,
* `spin_channels` &mdash; one scipp `DataGroup` per channel,
* `raw` &mdash; the full merged data group (for code that only understands the flat structure).

In [ ]:
polarized = load_polarized(DATA_FILE)

print('type        :', type(polarized).__name__)
print('polarization:', polarized.polarization)
print('spin_by_key :', polarized.spin_by_key)
print('channels    :', list(polarized.spin_channels))

In [ ]:
def channel_arrays(data_group):
    """Extract (q, R, dR, dq) numpy arrays from a single-channel scipp DataGroup."""
    data_key = list(data_group['data'])[0]
    coord_key = list(data_group['coords'])[0]
    q = data_group['coords'][coord_key].values
    dq = np.sqrt(data_group['coords'][coord_key].variances)
    r = data_group['data'][data_key].values
    dr = np.sqrt(data_group['data'][data_key].variances)
    return q, r, dr, dq


q_up, r_up, dr_up, dq_up = channel_arrays(polarized.spin_channels['spin_up'])
q_down, r_down, dr_down, dq_down = channel_arrays(polarized.spin_channels['spin_down'])

print('spin-up   q:', q_up, ' R:', r_up)
print('spin-down q:', q_down, ' R:', r_down)

In [ ]:
plt.errorbar(q_up, r_up, yerr=dr_up, fmt='o', label='spin-up (uu)', capsize=3)
plt.errorbar(q_down, r_down, yerr=dr_down, fmt='s', label='spin-down (dd)', capsize=3)
plt.xlabel('Q (1/Å)')
plt.ylabel('Reflectivity')
plt.title('Loaded spin channels (placeholder data)')
plt.legend()
plt.show()

## 2. Construct the model from the ORSO description

The ORSO header for this file stores the sample as a free-text **description**
(`air | material Ni, thickness 100 nm | Si`) rather than a machine-readable `model` block.

The sample loader (`LoadOrso` / `load_orso_model`) only builds a `Sample` automatically when the
file contains a proper `model` stack, so for this file it returns `None` and warns. We show that
below, then translate the description into a model by hand.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # silence orsopy's note about the raw 'p' polarization code
    parsed = orso.load_orso(DATA_FILE)

sample_info = parsed[0].info.data_source.sample
print('sample name       :', sample_info.name)
print('sample description:', sample_info.description)
print('sample.model      :', sample_info.model)

In [ ]:
# The sample loader cannot build a model from a free-text description, so it returns None here.
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    loaded_sample, _ = LoadOrso(parsed)

print('Sample built by the loader:', loaded_sample)

Translating the description `air | material Ni, thickness 100 nm | Si` into an EasyReflectometry
model: a vacuum/air superphase, a single **nickel** layer 100 nm (= 1000 Å) thick, and a silicon
subphase. Nickel is ferromagnetic, which makes it a natural choice for the polarized / magnetic
analysis below.

| Layer | Role | SLD (10⁻⁶ Å⁻²) | Thickness (Å) |
|-------|------|------------------|----------------|
| air   | superphase | 0.00 | 0 |
| Ni    | layer      | 9.41 | 1000 |
| Si    | subphase   | 2.07 | 0 |

In [ ]:
air = Material(sld=0.0, isld=0.0, name='air')
ni = Material(sld=9.41, isld=0.0, name='Ni')
si = Material(sld=2.07, isld=0.0, name='Si')

superphase = Multilayer(Layer(material=air, thickness=0, roughness=0, name='air'), name='Superphase')
ni_layer = Multilayer(Layer(material=ni, thickness=1000, roughness=5, name='Ni'), name='Ni layer')
subphase = Multilayer(Layer(material=si, thickness=0, roughness=5, name='Si'), name='Subphase')

sample = Sample(superphase, ni_layer, subphase, name=sample_info.name or 'Ni sample')
model = Model(sample=sample, scale=1, background=0, name='Ni model')
sample

## 3. Refl1d magnetic interface – both spin channels

We attach the Refl1d calculator and enable magnetism, then give the nickel layer a magnetic SLD
(`rhoM`) and in-plane angle (`thetaM`). Through the EasyReflectometry interface the magnetic
calculation returns the **spin-up** (`uu`) cross-section.

In [ ]:
interface = CalculatorFactory()
interface.switch('refl1d')
model.interface = interface
model.resolution_function = PercentageFwhm(0)

model_interface = model.interface()
model_interface.include_magnetism = True

# Assign magnetism to the Ni layer (the middle entry in the layer storage).
INITIAL_RHO_M = 1.5
INITIAL_THETA_M = 90.0
layer_keys = list(model_interface._wrapper.storage['layer'].keys())
ni_layer_key = layer_keys[1]
model_interface._wrapper.update_layer(ni_layer_key, magnetism_rhoM=INITIAL_RHO_M, magnetism_thetaM=INITIAL_THETA_M)

# A broad Q range shows the magnetic splitting (the file's own Q points sit in total reflection).
q_model = np.linspace(0.005, 0.15, 400)
easy_spin_up = model.interface().reflectity_profile(q_model, model.unique_name)
easy_spin_up[:5]

To obtain **both** cross-sections we use the same Refl1d machinery as `magnetism.ipynb`:
`_get_polarized_probe(..., all_polarizations=True)` returns the four cross-sections
`[uu, ud, du, dd]`. For a half-polarized measurement the spin-up channel corresponds to `uu`
(index 0) and spin-down to `dd` (index 3).

We build a parallel Refl1d sample mirroring the EasyReflectometry model (note Refl1d orders layers
subphase-first, the reverse of EasyReflectometry).

In [ ]:
refl1d_air = refl1d.names.SLD(name='air', rho=0.0, irho=0.0)
refl1d_ni = refl1d.names.SLD(name='Ni', rho=9.41, irho=0.0)
refl1d_si = refl1d.names.SLD(name='Si', rho=2.07, irho=0.0)


def magnetic_cross_sections(q_array, thickness, rho_m, theta_m):
    """Return (uu, dd) reflectivity for the magnetic Ni model via the Refl1d polarized probe."""
    refl1d_sample = (
        refl1d_si(0, 5)
        | refl1d_ni(thickness, 5, magnetism=refl1d.names.Magnetism(rhoM=rho_m, thetaM=theta_m))
        | refl1d_air(0, 0)
    )
    storage = {'model': {'m': {'scale': 1.0, 'bkg': 0.0}}}
    polarized_probe = _get_polarized_probe(
        q_array=q_array,
        dq_array=np.zeros_like(q_array),
        model_name='m',
        storage=storage,
        all_polarizations=True,
    )
    reflectivity = refl1d.names.Experiment(probe=polarized_probe, sample=refl1d_sample).reflectivity()
    uu = reflectivity[0][1]
    dd = reflectivity[3][1]
    return uu, dd


uu_model, dd_model = magnetic_cross_sections(q_model, 1000, INITIAL_RHO_M, INITIAL_THETA_M)

In [ ]:
plt.plot(q_model, uu_model, '-', label='spin-up  (uu)')
plt.plot(q_model, dd_model, '-', label='spin-down (dd)')
plt.plot(q_model, easy_spin_up, 'k--', lw=1, label='EasyReflectometry (uu)')
plt.xlabel('Q (1/Å)')
plt.ylabel('Reflectivity')
plt.yscale('log')
plt.title('Magnetic Ni model – two spin cross-sections')
plt.legend()
plt.show()

The EasyReflectometry magnetic interface (dashed) reproduces the Refl1d `uu` cross-section, and the
`uu`/`dd` curves separate where the magnetic contribution matters.

## 4. Try to fit both channels

Finally we *attempt* a simultaneous fit of the two spin channels. The magnetic Ni model has three
free parameters &mdash; the layer thickness and the magnetic SLD/angle &mdash; and the residual
stacks the `uu` mismatch against the spin-up data with the `dd` mismatch against the spin-down data.

> The reflectivities in this fixture are placeholders (`R > 1`, only a handful of points in the
> total-reflection region), so the fit will not converge to anything physical. It demonstrates the
> **mechanics** of driving the Refl1d magnetic interface against both channels at once.

In [ ]:
def residuals(params):
    thickness, rho_m, theta_m = params
    uu_up, _ = magnetic_cross_sections(q_up, thickness, rho_m, theta_m)
    _, dd_down = magnetic_cross_sections(q_down, thickness, rho_m, theta_m)
    return np.concatenate([uu_up - r_up, dd_down - r_down])


x0 = [1000.0, INITIAL_RHO_M, INITIAL_THETA_M]
bounds = ([100.0, 0.0, 0.0], [3000.0, 10.0, 180.0])

result = least_squares(residuals, x0, bounds=bounds, max_nfev=200)

print('converged :', result.success)
print('message   :', result.message)
print('thickness :', f'{result.x[0]:.1f} Å')
print('rhoM      :', f'{result.x[1]:.3f} (10⁻⁶ Å⁻²)')
print('thetaM    :', f'{result.x[2]:.1f} deg')
print('final cost:', f'{result.cost:.4g}')

In [ ]:
fit_thickness, fit_rho_m, fit_theta_m = result.x
uu_fit, dd_fit = magnetic_cross_sections(q_model, fit_thickness, fit_rho_m, fit_theta_m)

plt.errorbar(q_up, r_up, yerr=dr_up, fmt='o', label='spin-up data', capsize=3)
plt.errorbar(q_down, r_down, yerr=dr_down, fmt='s', label='spin-down data', capsize=3)
plt.plot(q_model, uu_fit, '-', label='spin-up fit (uu)')
plt.plot(q_model, dd_fit, '-', label='spin-down fit (dd)')
plt.xlabel('Q (1/Å)')
plt.ylabel('Reflectivity')
plt.title('Fitted magnetic model vs. both spin channels')
plt.legend()
plt.show()

## Summary

* `load_polarized` recognised the polarized ORSO file and returned a `PolarizedData` container with
  the two spin channels resolved from their `data_set` labels.
* The sample loader returned `None` because this file stores only a free-text description; we built
  the air | Ni | Si model by hand. A file with a proper ORSO `model` stack would be built
  automatically by `LoadOrso`.
* Using the Refl1d magnetic interface (`_get_polarized_probe`, as in `magnetism.ipynb`) we computed
  the `uu` and `dd` cross-sections and ran a simultaneous fit of both channels.

With a real polarized dataset (physical reflectivities over a useful Q range) the same workflow
would yield a meaningful magnetic fit.